In [7]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.request import urlretrieve
import time

In [8]:
# Function to scrape a single TV show poster
def scrape_tv_poster(imdb_url, save_dir):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }
        response = requests.get(imdb_url, headers=headers)
        response.raise_for_status()  # Raise an error for bad status codes
        
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')
        
       # Find the metadata for the type of content
        metadata = soup.find('script', type='application/ld+json')
        if not metadata:
            return False
        
        # Extract the JSON data
        import json
        data = json.loads(metadata.string)
        
        img_tag = data.get('image')
        if not img_tag:
            print(f"Image tag not found for {imdb_url}")
            return
        
        poster_url = img_tag
        
        # Extract the IMDb ID from the URL 
        imdb_id = imdb_url.split('/')[4]
        print(f"IMDb ID: {imdb_id}")
        
        # Create the save path
        save_path = os.path.join(save_dir, f"{imdb_id}.jpg")
        
        # Download and save the image
        urlretrieve(poster_url, save_path)
        print(f"Poster saved to {save_path}")
    
    except Exception as e:
        print(f"Error scraping {imdb_url}: {e}")

In [9]:
# Function to check if the IMDb URL is for a TV show
def is_tv_show(imdb_url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }
        response = requests.get(imdb_url, headers=headers)
        response.raise_for_status()  # Raise an error for bad status codes
        
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the metadata for the type of content
        metadata = soup.find('script', type='application/ld+json')
        if not metadata:
            return False
        
        # Extract the JSON data
        import json
        data = json.loads(metadata.string)
        
        # Check if the content type is a TV show
        if data.get("@type") == "TVSeries":
            return True
        return False
    
    except Exception as e:
        print(f"Error checking {imdb_url}: {e}")
        return False

In [10]:
# Function to scrape posters for a list of IMDb URLs
def scrape_posters_in_bulk(imdb_urls, save_dir, delay=1):
    # Ensure the save directory exists
    os.makedirs(save_dir, exist_ok=True)
    
    # Iterate through the list of IMDb URLs
    for i, imdb_url in enumerate(imdb_urls):
        # Check if the URL is for a TV show
        if is_tv_show(imdb_url):
            print(f"Scraping TV show poster: {imdb_url}")
            scrape_tv_poster(imdb_url, save_dir)
        else:
            print(f"Skipping non-TV show: {imdb_url}")
        
        # Add a delay to avoid overwhelming the server
        time.sleep(delay)

In [11]:
def printjson(imdb_url):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        }
        response = requests.get(imdb_url, headers=headers)
        response.raise_for_status()  # Raise an error for bad status codes
        
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the metadata for the type of content
        metadata = soup.find('script', type='application/ld+json')
        if not metadata:
            return False
        import json
        data = json.loads(metadata.string)
        print(json.dumps(data, indent=4))
    except Exception as e:
        print(f"Error checking {imdb_url}: {e}")
        return False

In [ ]:
# Example usage
if __name__ == "__main__":
    # Replace this list with your own IMDb URLs (up to 200,000)
    #imdb_urls = ["https://www.imdb.com/title/tt2098220/"]
    imdb_urls = [f"https://www.imdb.com/title/tt{str(i).zfill(7)}/" for i in range(6492, 7000000)]
    #printjson(imdb_urls[0])
    
    # Directory to save the posters
    save_dir = "tv_show_posters"
    
    # Scrape posters in bulk
    scrape_posters_in_bulk(imdb_urls, save_dir)